**Q-learning versus a Gecko A/B simulation baseline**

Run all cells on a CPU runtime. The experiment retrains both Q-learning variants over five seeds and evaluates six policies on identical held-out synthetic traces.
Paper grounding
Adapted from Sections III-D and III-E of Real-Time Video Gets a Fast Lane via Smart Queue Flushing at the Wireless Edge, IEEE INFOCOM 2026, DOI 10.1109/INFOCOM59046.2026.11571420.

The baseline implements initial keep/flush trials, comparison with a time-decayed average of the alternative action's bad-frame count, forced switching after five identical actual responses (subject to important-frame safety), and the router's five-alert flush-ratio threshold update. Threshold initialization is 100 ms, multiplicative factor 0.7, and additive increment 10 ms.

**Two A/B variants are included:**
- Gecko AB selective uses the same selective outdated-normal-frame flush as the RL agents. This controls the flush operation when comparing decision policies.
- Gecko AB atomic clears the entire flow queue and vetoes flushing when an important frame is queued, approximating the paper's queue operation and I-frame safeguard.

In [2]:
import argparse
import csv
from collections import deque
from dataclasses import dataclass
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

SLOT = 1 / 30
DEADLINE = 0.150
CAPACITY = 12
KEEP, FLUSH = 0, 1
SCENARIOS = ("Stable", "Fluctuating", "Sudden drop")
METRICS = ("Mean delay (ms)", "P95 delay (ms)", "Loss (%)", "On-time (%)")


@dataclass
class Frame:
    born: float
    bits: float
    important: bool


def make_trace(scenario, seed, length=300):
    rng = np.random.default_rng(seed)
    sizes = rng.integers(140_000, 240_001, length)
    important = np.arange(length) % 15 == 0
    if scenario == "Stable":
        rates = np.full(length, 8.0)
    elif scenario == "Fluctuating":
        rates = np.repeat(rng.choice([3, 6, 9], (length + 14) // 15), 15)[:length]
    else:
        rates = np.full(length, 9.0)
        rates[length // 3:2 * length // 3] = 2.0
    return sizes, important, rates * 1e6


class WirelessQueue:
    def __init__(self, trace):
        self.sizes, self.important, self.rates = trace
        self.queue = deque()
        self.index = 0
        self.delays = []
        self.dropped = 0
        self.delivered_by_class = [[], []]
        self.enqueue()

    def enqueue(self):
        if len(self.queue) >= CAPACITY:
            return 1
        i = self.index
        self.queue.append(Frame(i * SLOT, float(self.sizes[i]), self.important[i]))
        return 0

    def state(self):
        # Observations are taken after the current frame arrival.
        head = self.queue[0]
        count = len(self.queue)
        rate = self.rates[self.index]
        age = self.index * SLOT - head.born
        return (
            int(count >= 3) + int(count >= 7),
            int(rate >= 4e6) + int(rate >= 7e6),
            int(age >= 0.050) + int(age >= 0.100),
            int(head.important),
        )

    def gecko_action(self):
        # Threshold baseline using predicted completion of the newest queued frame.
        backlog = sum(frame.bits for frame in self.queue)
        tail_age = self.index * SLOT - self.queue[-1].born
        return FLUSH if tail_age + backlog / self.rates[self.index] > DEADLINE else KEEP

    def flush(self, now, rate):
        kept = deque()
        bits_ahead = dropped = useful = 0
        for frame in self.queue:
            age = now - frame.born
            prediction = age + (bits_ahead + frame.bits) / rate
            if not frame.important and prediction > DEADLINE:
                dropped += 1
                # An unexpired frame could still be useful if bandwidth recovers.
                useful += int(age < DEADLINE)
            else:
                kept.append(frame)
                bits_ahead += frame.bits
        self.queue = kept
        return dropped, useful

    def transmit(self, now, rate, duration):
        delivered = []
        budget = rate * duration
        while self.queue and budget > 0:
            head = self.queue[0]
            if head.bits > budget:
                head.bits -= budget
                break
            budget -= head.bits
            now += head.bits / rate
            frame = self.queue.popleft()
            delay = now - frame.born
            delivered.append(delay)
            self.delivered_by_class[int(frame.important)].append(delay)
        return delivered

    def step(self, action):
        now = self.index * SLOT
        rate = self.rates[self.index]
        flushed, useful = self.flush(now, rate) if action == FLUSH else (0, 0)
        self.index += 1
        done = self.index == len(self.sizes)
        # Hold the final rate while draining, with no more arrivals.
        duration = SLOT
        if done:
            duration += sum(frame.bits for frame in self.queue) / rate
        delivered = self.transmit(now, rate, duration)
        overflow = 0 if done else self.enqueue()

        self.delays.extend(delivered)
        self.dropped += flushed + overflow
        resolved = max(1, len(delivered) + flushed + overflow)
        delay = sum(delivered) / (DEADLINE * resolved)
        loss = (useful + overflow) / resolved
        missed = (flushed + overflow + sum(d > DEADLINE for d in delivered)) / resolved
        reward = -(0.5 * delay + 0.5 * loss + 2.0 * missed)
        return None if done else self.state(), reward


# Both agents use the same environment, reward, and importance-aware flush rule.
# Only the importance observation is removed in the 27-state agent.
def observe(state, use_importance):
    return state if use_importance else state[:3]


def train(episodes, seed, use_importance):
    rng = np.random.default_rng(seed)
    shape = (3, 3, 3, 2, 2) if use_importance else (3, 3, 3, 2)
    q = np.zeros(shape)
    history = []
    for episode in range(episodes):
        scenario = SCENARIOS[episode % 3]
        # Independent stream from evaluation, paired across the two variants.
        trace_seed = np.random.SeedSequence([1, seed, episode])
        env = WirelessQueue(make_trace(scenario, trace_seed))
        state = env.state()
        total = 0.0
        epsilon = max(0.05, 0.995 ** episode)
        while state is not None:
            observation = observe(state, use_importance)
            if rng.random() < epsilon:
                action = int(rng.integers(2))
            else:
                action = int(np.argmax(q[observation]))
            next_state, reward = env.step(action)
            future = 0.0
            if next_state is not None:
                future = np.max(q[observe(next_state, use_importance)])
            entry = observation + (action,)
            q[entry] += 0.10 * (reward + 0.95 * future - q[entry])
            total += reward
            state = next_state
        history.append(total / len(env.sizes))
    return q, history


COLUMNS = (
    "Mean delay ms", "P95 delay ms", "Loss pct", "On-time pct",
    "Important on-time pct", "Normal on-time pct",
    "Important loss pct", "Normal loss pct",
)


def evaluate(policy, traces, controller_factory=None):
    delivered = [[], []]
    generated = np.zeros(2, dtype=int)
    for trace in traces:
        env = WirelessQueue(trace)
        active_policy = controller_factory() if controller_factory else policy
        state = env.state()
        while state is not None:
            state, _ = env.step(active_policy(env))
            if controller_factory:
                active_policy.after_step(env)
        assert len(env.delays) + env.dropped == len(env.sizes)
        for group in (0, 1):
            generated[group] += np.count_nonzero(env.important == group)
            delivered[group].extend(env.delivered_by_class[group])

    delays = delivered[0] + delivered[1]
    on_time = np.array([sum(d <= DEADLINE for d in group) for group in delivered])
    losses = generated - np.array([len(group) for group in delivered])
    return np.array([
        np.mean(delays) * 1000 if delays else np.nan,
        np.percentile(delays, 95) * 1000 if delays else np.nan,
        100 * losses.sum() / generated.sum(),
        100 * on_time.sum() / generated.sum(),
        100 * on_time[1] / generated[1],
        100 * on_time[0] / generated[0],
        100 * losses[1] / generated[1],
        100 * losses[0] / generated[0],
    ])



class GeckoAB:
    """Simulation adaptation of Sections III-D/E, with explicit assumptions.

    Alert outcomes are counted during a 200 ms minimum observation window,
    extended to the next eligible alert. EMA time constant is 1 second.
    These timing choices and threshold bounds are simulation assumptions.
    """

    def __init__(self, atomic=False):
        self.atomic = atomic
        self.threshold = 0.100
        self.responses = deque(maxlen=5)
        self.old_ratio = 0.0
        self.averages = [None, None]
        self.next_action = KEEP
        self.previous = None
        self.bad = 0
        self.started = 0
        self.last_action = None
        self.streak = 0
        self.seen_delivered = [0, 0]
        self.seen_dropped = 0
        self.events = []

    def finish_interval(self, now):
        if self.previous is None:
            return
        action = self.previous
        alternative = self.averages[1 - action]
        # Compare the latest count against the alternative's historical average.
        if alternative is not None and self.bad > alternative:
            self.next_action = 1 - action
        else:
            self.next_action = action
        decay = np.exp(-(now - self.started) / 1.0)
        old = self.averages[action]
        self.averages[action] = self.bad if old is None else decay * old + (1 - decay) * self.bad

    def __call__(self, env):
        now = env.index * SLOT
        # Suppress repeated alerts during the assumed response/observation window.
        if self.previous is not None and now - self.started < 0.200 - 1e-9:
            return KEEP
        prediction = sum(frame.bits for frame in env.queue) / env.rates[env.index]
        if prediction <= self.threshold:
            return KEEP
        self.finish_interval(now)
        if not self.events:
            requested = KEEP
        elif len(self.events) == 1:
            requested = FLUSH
        else:
            requested = self.next_action
        if self.streak >= 5:
            requested = 1 - self.last_action

        # Atomic queue flushing is vetoed if it would discard an important frame.
        # Selective mode retains the existing simulator's important-frame protection.
        action = requested
        if self.atomic and any(frame.important for frame in env.queue):
            action = KEEP

        self.streak = self.streak + 1 if action == self.last_action else 1
        self.last_action = action
        self.previous = action
        self.started = now
        self.bad = 0
        self.responses.append(action)
        ratio = sum(self.responses) / len(self.responses)
        if ratio > 0.5 and ratio > self.old_ratio:
            self.threshold = max(0.020, 0.7 * self.threshold)
        elif ratio < 0.5 and ratio < self.old_ratio:
            self.threshold = min(0.500, self.threshold + 0.010)
        self.old_ratio = ratio
        self.events.append((now, requested, action, self.threshold))

        if self.atomic and action == FLUSH:
            env.dropped += len(env.queue)
            env.queue.clear()
            return KEEP  # Atomic operation already executed before transmission.
        return action

    def after_step(self, env):
        # Observe only events that have occurred, with no lookahead or cloned trials.
        bad = env.dropped - self.seen_dropped
        self.seen_dropped = env.dropped
        for group in (0, 1):
            new = env.delivered_by_class[group][self.seen_delivered[group]:]
            bad += sum(delay > DEADLINE for delay in new)
            self.seen_delivered[group] = len(env.delivered_by_class[group])
        if self.previous is not None:
            self.bad += bad


def run_experiment(episodes=600, seeds=5, output="gecko_results"):
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    traces = {
        scenario: [make_trace(scenario, np.random.SeedSequence([2, i])) for i in range(20)]
        for scenario in SCENARIOS
    }
    records = []
    histories = {}
    for enabled, name in ((True, "Q with importance"), (False, "Q without importance")):
        histories[name] = []
        for seed in range(seeds):
            q, history = train(episodes, seed, enabled)
            histories[name].append(history)
            np.save(output / f"q_{enabled}_seed_{seed}.npy", q)
            for scenario in SCENARIOS:
                def policy(env):
                    return int(np.argmax(q[observe(env.state(), enabled)]))
                scores = evaluate(policy, traces[scenario])
                records.append((scenario, name, seed, *scores))
            print(f"Trained {name}, seed {seed}", flush=True)
    for scenario in SCENARIOS:
        for name, policy in (
            ("FIFO", lambda env: KEEP),
            ("Fixed threshold", lambda env: env.gecko_action()),
        ):
            records.append((scenario, name, "", *evaluate(policy, traces[scenario])))
        for atomic, name in (
            (False, "Gecko AB selective"),
            (True, "Gecko AB atomic"),
        ):
            scores = evaluate(None, traces[scenario], lambda: GeckoAB(atomic=atomic))
            records.append((scenario, name, "", *scores))

    import pandas as pd
    raw = pd.DataFrame(records, columns=["Scenario", "Policy", "Seed", *COLUMNS])
    raw.to_csv(output / "per_seed_metrics.csv", index=False)
    means = raw.groupby(["Scenario", "Policy"], sort=False)[list(COLUMNS)].mean()
    std = raw.groupby(["Scenario", "Policy"], sort=False)[list(COLUMNS)].std().fillna(0)
    summary = means.add_suffix(" mean").join(std.add_suffix(" std"))
    summary.to_csv(output / "summary.csv")
    print(means[["On-time pct", "P95 delay ms", "Loss pct"]].round(2).to_string())

    names = list(dict.fromkeys(raw["Policy"]))
    fig, axes = plt.subplots(2, 2, figsize=(14, 9), layout="constrained")
    x = np.arange(3)
    width = 0.13
    for ax, column in zip(axes.flat, ("On-time pct", "P95 delay ms",
                                     "Important on-time pct", "Loss pct")):
        for i, name in enumerate(names):
            values = [means.loc[(scenario, name), column] for scenario in SCENARIOS]
            errors = [std.loc[(scenario, name), column] for scenario in SCENARIOS]
            ax.bar(x + (i - 2.5) * width, values, width,
                   yerr=errors, capsize=2, label=name)
        ax.set_xticks(x, SCENARIOS)
        ax.set_title(column)
    axes[0, 0].legend(fontsize=8)
    fig.savefig(output / "gecko_comparison.png", dpi=170)
    plt.close(fig)

    # Save an alert audit for one held-out fluctuating trace for each AB variant.
    audits = []
    for atomic in (False, True):
        env = WirelessQueue(traces["Fluctuating"][0])
        controller = GeckoAB(atomic)
        state = env.state()
        while state is not None:
            state, _ = env.step(controller(env))
            controller.after_step(env)
        for now, requested, actual, threshold in controller.events:
            audits.append((atomic, now, requested, actual, threshold))
    pd.DataFrame(audits, columns=["Atomic flush", "Time seconds", "Requested action",
                                 "Actual action", "Threshold seconds"]).to_csv(
        output / "alert_audit.csv", index=False)
    return summary

**Train and compare**

Keep the defaults for the five-seed experiment.

In [3]:
EPISODES = 600
SEEDS = 5
OUTPUT = "gecko_results"
summary = run_experiment(EPISODES, SEEDS, OUTPUT)


Trained Q with importance, seed 0
Trained Q with importance, seed 1
Trained Q with importance, seed 2
Trained Q with importance, seed 3
Trained Q with importance, seed 4
Trained Q without importance, seed 0
Trained Q without importance, seed 1
Trained Q without importance, seed 2
Trained Q without importance, seed 3
Trained Q without importance, seed 4
                                  On-time pct  P95 delay ms  Loss pct
Scenario    Policy                                                   
Stable      Q with importance          100.00         29.37      0.00
Fluctuating Q with importance           81.00        162.25     13.14
Sudden drop Q with importance           72.01        163.18     23.03
Stable      Q without importance       100.00         29.37      0.00
Fluctuating Q without importance        81.48        160.09     13.03
Sudden drop Q without importance        72.82        159.35     22.75
Stable      FIFO                       100.00         29.37      0.00
            Fix

**Interpret the comparison**

Compare on-time delivery, P95 delay, and loss together. Delay describes delivered frames, while on-time and loss rates use all generated frames.

Use the selective A/B comparison to assess decision logic under a shared flush rule. The atomic variant changes both decision logic and flush semantics. The zero standard deviations of deterministic baselines reflect a single fixed evaluation set and do not establish certainty across network traces. RL error bars measure training-seed variability.

The alert audit records actual and requested actions plus the adapted threshold. A veto can prevent the intended exploration action.

In [4]:
import pandas as pd
display(pd.read_csv(f"{OUTPUT}/alert_audit.csv").head(20))

,Atomic flush,Time seconds,Requested action,Actual action,Threshold seconds
0,False,0.066667,0,0,0.10
1,False,0.266667,1,1,0.10
2,False,0.466667,0,0,0.11
3,False,1.533333,0,0,0.12
4,False,1.733333,0,0,0.13
5,False,1.933333,0,0,0.13
6,False,2.133333,1,1,0.13
7,False,2.333333,0,0,0.13
8,False,2.533333,0,0,0.13
9,False,2.733333,0,0,0.13


In [5]:
import shutil
from google.colab import files
shutil.make_archive("gecko_results", "zip", OUTPUT)
files.download("gecko_results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>